# 🏭 ApexInspect AI — Huấn Luyện Custom YOLOv8 PCB Defect Detection
Notebook này được thiết kế để chạy trực tiếp trên **Google Colab (qua VS Code Extension hoặc Trình duyệt)** với GPU T4 miễn phí.

**Mục tiêu:**
1. Tải dataset lỗi bo mạch thực tế từ **Kaggle** (`akhatova/pcb-defects`) hoặc **Roboflow Universe** (6 classes).
2. Fine-tune mô hình `yolov8n.pt` trong 50 epochs.
3. Đánh giá độ chính xác: mAP@0.5, mAP@0.5:0.95, Confusion Matrix.
4. Tối ưu hóa và xuất mô hình sang **ONNX Runtime** phục vụ Edge Deployment.

## Bước 1: Kiểm Tra GPU & Cài Đặt Môi Trường

In [ ]:
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime kagglehub

## Bước 2: Tải Dataset Thật Từ Kaggle hoặc Roboflow

Bộ dữ liệu gồm 6 loại lỗi thực tế trong dây chuyền sản xuất bo mạch SMT:
- `missing_hole`: Mất lỗ khoan xuyên bo mạch
- `mouse_bite`: Khuyết vết cắn trên đường mạch đồng
- `open_circuit`: Đứt mạch đồng dẫn điện
- `short`: Hàn chập chân linh kiện
- `spur`: Râu đồng thừa
- `spurious_copper`: Vết ố đồng dư thừa ngoài ý muốn

> **Lựa chọn 1**: Dùng thư viện chính thức `kagglehub` (Google/Kaggle) — Tải trực tiếp không cần API key thủ công.
> **Lựa chọn 2**: Tải từ Roboflow Universe (đã format sẵn chuẩn YOLOv8).

In [ ]:
import os
import shutil

# ==============================================================================
# PHƯƠNG ÁN 1: Tải từ Kaggle bằng thư viện chính thức kagglehub
# Link dataset gốc: https://www.kaggle.com/datasets/akhatova/pcb-defects
# ==============================================================================
try:
    import kagglehub
    print("📥 Đang tải dataset PCB Defect từ Kaggle (akhatova/pcb-defects)...")
    download_path = kagglehub.dataset_download("akhatova/pcb-defects")
    print(f"✅ Tải thành công về: {download_path}")
except Exception as e:
    print(f"ℹ️ Gợi ý: {e}")
    print("Bạn có thể tải thủ công tại: https://www.kaggle.com/datasets/akhatova/pcb-defects")
    print("Hoặc từ Roboflow: https://universe.roboflow.com/mohamed-traore-2wbn5/pcb-defects-r8spd")

# Cấu hình file data.yaml chuẩn hóa cho YOLOv8
yaml_content = """
path: ./dataset_pcb
train: images/train
val: images/val
test: images/val

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content.strip())
print("✅ Đã khởi tạo cấu hình data.yaml thành công!")

## Bước 3: Huấn Luyện Mô Hình YOLOv8 Nano
Lựa chọn `yolov8n.pt` vì tính cơ động, độ trễ cực thấp (< 35ms trên CPU), rất phù hợp với bài toán Edge Vision trên dây chuyền sản xuất.

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình nền tảng YOLOv8n tiền huấn luyện
model = YOLO("yolov8n.pt")

# Bắt đầu huấn luyện
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,         # 0 cho GPU Colab, 'cpu' nếu chạy CPU, 'mps' nếu chạy trên Mac
    optimizer="AdamW",
    lr0=0.001,
    mosaic=1.0,
    project="runs/apex_inspect",
    name="yolov8n_pcb_custom"
)

print("🎉 Quá trình huấn luyện đã hoàn tất!")

## Bước 4: Đánh Giá & Trực Quan Hóa Kết Quả (Validation & Metrics)

In [ ]:
from IPython.display import Image, display

# Hiển thị ma trận nhầm lẫn (Confusion Matrix)
conf_matrix_path = "runs/apex_inspect/yolov8n_pcb_custom/confusion_matrix.png"
if os.path.exists(conf_matrix_path):
    print("📊 Ma trận nhầm lẫn (Confusion Matrix):")
    display(Image(filename=conf_matrix_path))

# Hiển thị biểu đồ kết quả mAP và Loss
results_path = "runs/apex_inspect/yolov8n_pcb_custom/results.png"
if os.path.exists(results_path):
    print("📈 Đồ thị huấn luyện (Loss & mAP Curves):")
    display(Image(filename=results_path))

## Bước 5: Tối Ưu Hóa & Xuất Sang Định Dạng ONNX (Edge Deployment)
Chuyển đổi trọng số `best.pt` sang định dạng `ONNX` kèm tối ưu hóa đồ thị tính toán (Graph Optimization).

In [ ]:
# Load trọng số tốt nhất đã huấn luyện
best_model_path = "runs/apex_inspect/yolov8n_pcb_custom/weights/best.pt"
custom_model = YOLO(best_model_path if os.path.exists(best_model_path) else "yolov8n.pt")

# Xuất sang ONNX
onnx_path = custom_model.export(format="onnx", imgsz=640, optimize=True, half=False)
print(f"✅ Mô hình ONNX đã xuất thành công tại: {onnx_path}")

# Copy mô hình vào thư mục models/ của dự án ApexInspect AI
target_onnx = "../../models/yolov8n_pcb_defect.onnx"
if os.path.exists(os.path.dirname(target_onnx)):
    shutil.copy(onnx_path, target_onnx)
    print(f"🚀 Đã cập nhật mô hình vào dự án: {target_onnx}")
else:
    print(f"Tải file {onnx_path} về và đặt vào thư mục models/yolov8n_pcb_defect.onnx")